# 🔬 Skin Cancer Detection — Model Training Notebook

This notebook trains **6 models** for skin cancer classification on the HAM10000 dataset:

| # | Model | Type | Architecture Concept |
|---|---|---|---|
| 1 | Sequential CNN | From scratch | Simple conv stacking |
| 2 | Custom ResNet | From scratch | Residual connections |
| 3 | EfficientNetB0 | Transfer learning | Compound scaling |
| 4 | ResNet50 | Transfer learning | Additive residuals (deep) |
| 5 | DenseNet121 | Transfer learning | Dense feature reuse |
| 6 | ViT (Vision Transformer) | Transfer learning | Self-attention |

Plus **class imbalance experiment** (EfficientNet + class weights).

**Runtime**: ~3.5 hours total on T4 GPU.

---

## Cell 1: GPU Check & Environment Setup

In [ ]:
# Verify GPU is available (should show Tesla T4)
!nvidia-smi
print("\n" + "="*60)

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {gpus}")

if not gpus:
    print("\n⚠️  NO GPU DETECTED!")
    print("Go to: Runtime → Change runtime type → T4 GPU → Save")
else:
    print("\n✅ GPU is ready!")

## Cell 2: Clone Project Repository

In [ ]:
import os

# Clone the repository
if not os.path.exists('/content/SkinCancer'):
    !git clone https://github.com/Ashutosh-Repos/SkinCancer.git /content/SkinCancer
    print("✅ Repository cloned!")
else:
    !cd /content/SkinCancer && git pull
    print("✅ Repository updated!")

# Change to project directory
%cd /content/SkinCancer

# Verify project structure
!ls -la src/

## Cell 3: Install Dependencies

In [ ]:
# Most packages (tensorflow, numpy, pandas, sklearn, matplotlib) are pre-installed on Colab
# Only install project-specific packages
!pip install flask flask-cors python-dotenv tqdm tensorflow-hub -q

print("\n✅ Dependencies installed!")

# Setup Python path so imports work
import sys
sys.path.insert(0, '/content/SkinCancer/src')

# Verify imports
from config import DATASET_CONFIG, MODEL_CONFIG, TRAINING_CONFIG, TRANSFER_LEARNING_MODELS
print(f"Default image size: {DATASET_CONFIG['image_size']}")
print(f"Test split: {TRAINING_CONFIG['test_split']}")
print(f"Available models: {list(MODEL_CONFIG.keys())}")
print(f"Transfer learning models: {TRANSFER_LEARNING_MODELS}")

## Cell 4: Download HAM10000 Dataset

**Option A**: Using Kaggle API (faster, ~3 min)  
**Option B**: Upload from Google Drive

In [ ]:
# ============================================================
# OPTION A: Kaggle API (Recommended)
# ============================================================
# Upload your kaggle.json file when prompted

import os

if not os.path.exists('data/images') or len(os.listdir('data/images')) < 10000:
    print("Dataset not found. Downloading...")
    print("\n📁 Please upload your kaggle.json file:")
    
    from google.colab import files
    uploaded = files.upload()  # Upload kaggle.json
    
    !mkdir -p ~/.kaggle
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    
    # Download dataset
    !python scripts/download_dataset.py
    
    print(f"\n✅ Dataset downloaded!")
else:
    print("✅ Dataset already exists!")

# Verify
image_count = len([f for f in os.listdir('data/images') if f.endswith('.jpg')])
print(f"Images found: {image_count}")
!head -3 data/HAM10000_metadata.csv

## Cell 5: Mount Google Drive (for persistent output)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create output directories on Drive
DRIVE_DIR = '/content/drive/MyDrive/SkinCancer'
for subdir in ['models/checkpoints', 'logs', 'results', 'results/gradcam']:
    os.makedirs(f'{DRIVE_DIR}/{subdir}', exist_ok=True)

print(f"\n✅ Google Drive mounted! Outputs will be saved to: {DRIVE_DIR}")

## Cell 6: Create Output Directories

In [ ]:
# Ensure all output directories exist
for d in ['models', 'models/checkpoints', 'logs', 'results', 'results/gradcam']:
    os.makedirs(d, exist_ok=True)

print("✅ Output directories ready")

---
## 🏋️ Model Training

Train all 6 models sequentially. Each cell is independent — you can run them in any order.

---

### Model 1: Sequential CNN (From Scratch) — ~15 min

In [ ]:
!cd /content/SkinCancer && python src/train.py --model sequential --epochs 30 --batch-size 64
print("\n✅ Sequential CNN training complete!")

### Model 2: Custom ResNet (From Scratch) — ~15 min

In [ ]:
!cd /content/SkinCancer && python src/train.py --model resnet --epochs 16 --batch-size 64
print("\n✅ Custom ResNet training complete!")

### Model 3: EfficientNetB0 (Transfer Learning) — ~35 min
Two-stage training: Stage 1 (frozen base, 10 epochs) → Stage 2 (fine-tune, 20 epochs)

In [ ]:
!cd /content/SkinCancer && python src/train.py --model efficientnet --batch-size 32
print("\n✅ EfficientNetB0 training complete!")

### Model 4: ResNet50 (Transfer Learning) — ~30 min

In [ ]:
!cd /content/SkinCancer && python src/train.py --model resnet50 --batch-size 32
print("\n✅ ResNet50 training complete!")

### Model 5: DenseNet121 (Transfer Learning) — ~30 min

In [ ]:
!cd /content/SkinCancer && python src/train.py --model densenet --batch-size 32
print("\n✅ DenseNet121 training complete!")

### Model 6: Vision Transformer (ViT) — ~45 min
Uses ViT-B/16 from TensorFlow Hub pretrained on ImageNet-21k

In [ ]:
!cd /content/SkinCancer && python src/train.py --model vit --batch-size 16
print("\n✅ Vision Transformer training complete!")

---
## 📊 Class Imbalance Experiment

Re-train the best model (EfficientNet) with **class weights** to improve rare class performance.

---

### EfficientNetB0 + Class Weights — ~35 min

In [ ]:
# Rename previous EfficientNet checkpoint before training with class weights
import shutil
if os.path.exists('models/checkpoints/efficientnet_best.h5'):
    shutil.copy(
        'models/checkpoints/efficientnet_best.h5',
        'models/checkpoints/efficientnet_no_cw_best.h5'
    )
    if os.path.exists('models/checkpoints/efficientnet_best_metadata.json'):
        shutil.copy(
            'models/checkpoints/efficientnet_best_metadata.json',
            'models/checkpoints/efficientnet_no_cw_best_metadata.json'
        )
    print("Previous EfficientNet checkpoint backed up")

!cd /content/SkinCancer && python src/train.py --model efficientnet --batch-size 32 --class-weights
print("\n✅ EfficientNet + Class Weights training complete!")

---
## 📈 Evaluate All Models

Run evaluation on every trained model to generate:
- Confusion matrices
- Classification reports
- Per-class metrics

---

In [ ]:
import glob

# Find all trained model checkpoints
model_files = glob.glob('models/checkpoints/*_best.h5')
print(f"Found {len(model_files)} trained models:\n")
for f in sorted(model_files):
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f"  {os.path.basename(f):40s} ({size_mb:.1f} MB)")

# Evaluate each model
print("\n" + "="*60)
print("EVALUATING ALL MODELS")
print("="*60)

for model_file in sorted(model_files):
    model_name = os.path.basename(model_file)
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    !cd /content/SkinCancer && python src/evaluate.py --model {model_file}

## 📊 Model Comparison Table

In [ ]:
import json
import pandas as pd

# Collect results from all evaluated models
results = []
for metrics_file in sorted(glob.glob('results/*/evaluation_metrics.json')):
    model_name = metrics_file.split('/')[-2]
    with open(metrics_file) as f:
        m = json.load(f)
    results.append({
        'Model': model_name,
        'Accuracy': f"{m['accuracy']*100:.2f}%",
        'F1 (Macro)': f"{m['f1_macro']*100:.2f}%",
        'F1 (Weighted)': f"{m['f1_weighted']*100:.2f}%",
        'Precision (Macro)': f"{m['precision_macro']*100:.2f}%",
        'Recall (Macro)': f"{m['recall_macro']*100:.2f}%",
    })

if results:
    df = pd.DataFrame(results)
    print("\n" + "="*80)
    print("MODEL COMPARISON TABLE")
    print("="*80)
    display(df)
    
    # Save as CSV
    df.to_csv('results/model_comparison.csv', index=False)
    print("\n✅ Comparison table saved to results/model_comparison.csv")
else:
    print("No evaluation results found. Make sure to evaluate models first.")

## 📊 Display Training Curves & Confusion Matrices

In [ ]:
from IPython.display import Image as IPImage, display

# Show training history plots
print("\n" + "="*60)
print("TRAINING HISTORY PLOTS")
print("="*60)

log_dirs = sorted(glob.glob('logs/*/training_history.png'))
for plot_path in log_dirs:
    model_name = plot_path.split('/')[-2]
    print(f"\n--- {model_name} ---")
    display(IPImage(filename=plot_path, width=800))

# Show confusion matrices
print("\n" + "="*60)
print("CONFUSION MATRICES")
print("="*60)

cm_files = sorted(glob.glob('results/*/confusion_matrix.png'))
for cm_path in cm_files:
    model_name = cm_path.split('/')[-2]
    print(f"\n--- {model_name} ---")
    display(IPImage(filename=cm_path, width=800))

---
## 💾 Save Everything to Google Drive

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/SkinCancer'

# Copy all outputs to Drive
!cp -r models/checkpoints/ {DRIVE_DIR}/models/checkpoints/
!cp -r logs/ {DRIVE_DIR}/logs/
!cp -r results/ {DRIVE_DIR}/results/
!cp data/norm_stats.json {DRIVE_DIR}/ 2>/dev/null || true

print("\n✅ All outputs saved to Google Drive!")
print(f"   Location: {DRIVE_DIR}")
print("\nFiles saved:")
!find {DRIVE_DIR} -name '*.h5' -o -name '*.json' -o -name '*.png' -o -name '*.csv' | head -30

## 📥 Download Best Model to Your Computer

In [ ]:
from google.colab import files

# Download the best model (EfficientNet with class weights)
print("Downloading best model...")
best_model = 'models/checkpoints/efficientnet_best.h5'
best_metadata = 'models/checkpoints/efficientnet_best_metadata.json'

if os.path.exists(best_model):
    files.download(best_model)
if os.path.exists(best_metadata):
    files.download(best_metadata)
if os.path.exists('results/model_comparison.csv'):
    files.download('results/model_comparison.csv')

print("\n✅ Download complete!")
print("\nTo use on your MacBook:")
print("  python src/inference.py --model models/checkpoints/efficientnet_best.h5 --image <your_image.jpg>")